# Lesson 17 — 让状态实时推到页面

这节课仍然延续桌面状态提醒器，但数据首次穿过**真实本机网络**。不需要 ESP32，也不需要 Python 基础：下面代码格是完整 C++ 程序。

## 我们要解决什么问题？

轮询 HTTP 状态太频繁时，怎样让浏览器与设备保持一条双向连接？

## 最小知识

WebSocket 先用 HTTP Upgrade 建立连接，再交换带 opcode、长度和客户端 mask 的帧。本课的 C++ server 真正完成 101 握手、mask 解码和 text frame；为保持聚焦，只支持短文本帧。

运行初始化格后，所有网络实验只绑定 `127.0.0.1`。Notebook 面板是真实客户端；业务逻辑、协议响应和状态都来自当前 C++ 代码格。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


## 1. 第一个可运行实验

先发送 status frame，再发送 toggle frame，最后再发送 status。注意自动检查会创建新连接读取状态，确认设备状态没有只存在于某一个 WebSocket 客户端中。

运行下一格。面板的端口由 C++ 程序通过 `ACADEMY_READY` 报告；不要手填固定端口。

In [ ]:
%%network_lab l17-guided ws guided pass
#include "academy/net/websocket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_websocket_server([&](const std::string& message) {
        if (message == "status") {
            return std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}");
        }
        if (message == "toggle") {
            reminder_on = !reminder_on;
            return std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}");
        }
        return std::string("{\"error\":\"unknown command\"}");
    });
}


## 2. 修改一次

把下一格中一个响应文本或状态初值改掉，再点击面板操作，观察改变从 C++ 穿过真实 localhost 连接回到输出。重新运行本格会终止旧进程并启动新进程。

In [ ]:
%%network_lab l17-modify ws
#include "academy/net/websocket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_websocket_server([&](const std::string& message) {
        if (message == "status") {
            return std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}");
        }
        if (message == "toggle") {
            reminder_on = !reminder_on;
            return std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}");
        }
        return std::string("{\"error\":\"unknown command\"}");
    });
}


## 3. 故意弄坏

下面代码可编译，但**自动检查应失败**。先运行，点击“播放自动检查”，再根据红色结果解释问题。失败来自真实网络往返后的行为差异，不是 Notebook 预设动画。

In [ ]:
%%network_lab l17-broken ws exercise fail
#include "academy/net/websocket.hpp"

int main() {
    return academy::net::run_websocket_server([](const std::string& message) {
        bool reminder_on = false; // 每一帧都重置，不能保存设备状态。
        if (message == "toggle") reminder_on = true;
        return std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}");
    });
}


## 4. 自己完成

在下方 C++ 中完成 TODO。先手动操作，再点“播放自动检查”。测试只编译这一格，不会读取 `solution/`。需要时再看 [分层 Hint](../lessons/17-websocket-live-status/hints.md)。

In [ ]:
%%network_lab l17-exercise ws exercise
#include "academy/net/websocket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_websocket_server([&](const std::string& message) {
        // TODO: status 返回当前 JSON；toggle 翻转并返回更新后的 JSON。
        (void)message;
        (void)reminder_on;
        return std::string("{\"error\":\"TODO\"}");
    });
}


## 5. 你刚刚真正学到了什么？

你已经让提醒器通过真实 localhost 网络与另一个进程通信，并使用协议契约验证状态。ESP32 的 WebSocket server 需要考虑多个客户端、广播策略、最大 frame、心跳和断开清理。协议选择应由实时性和客户端数量决定，不是 HTTP 的“高级版”。

完成尝试后才查看 [Solution Notebook](solutions/17-websocket-live-status-solution.ipynb)。